# 02 · Camada Bronze

Ingere cada fonte como ela veio da origem. 

In [0]:
%pip install openpyxl
dbutils.library.restartPython()

## 1. Parâmetros

O arquivo do DEE e a planilha do Sebrae não têm URL estável e são carregados manualmente no volume.

In [0]:
import requests, urllib3, os, json, re, unicodedata
from datetime import datetime
from pyspark.sql.functions import lit
from pyspark.sql import functions as F

urllib3.disable_warnings(urllib3.exceptions.InsecureRequestWarning)

CATALOGO = "mvp_pipeline_vf"
VOL = f"/Volumes/{CATALOGO}/bronze/arquivos_brutos"
INGESTAO_TS = datetime.now().isoformat(timespec="seconds")

# (tabela_destino, subpasta, nome_arquivo, url)
FONTES_CSV = [
    ("icms_cnae_subclasse", "sefaz", "arrecadacao_icms_cnae_subclasse.csv",
     "https://receitadados.sefaz.rs.gov.br/Arquivos/Arrecada%C3%A7%C3%A3o%20de%20ICMS%20por%20CNAE%20-%20Subclasse.csv"),
    ("arrecadacao_municipio_corede", "sefaz", "arrecadacao_municipio_corede.csv",
     "https://receitadados.sefaz.rs.gov.br/Arquivos/Arrecada%C3%A7%C3%A3o%20por%20Munic%C3%ADpio%20e%20por%20Corede.csv"),
    ("desoneracoes", "sefaz", "desoneracoes_rs_completa.csv",
     "https://receitadados.sefaz.rs.gov.br/Arquivos/Desoneracoes_RS_completa.csv"),
    ("cadastro_contribuintes_setor", "sefaz", "cadastro_contribuintes_setor.csv",
     "https://receitadados.sefaz.rs.gov.br/Arquivos/Cadastro%20Contribuintes%20-%20Setor.csv"),
    ("cadastro_contribuintes_municipio", "sefaz", "cadastro_contribuintes_municipio.csv",
     "https://receitadados.sefaz.rs.gov.br/Arquivos/Cadastro%20Contribuintes%20-%20Municipio.csv"),
]

# Arquivos obtidos por download manual: o portal do DEE e a planilha do Sebrae
# não expõem URL estável de arquivo.
ARQUIVO_DEE = f"{VOL}/deers/19092428-pib-municipios-rs-2002-2023-serie-historica(dados).csv"
ARQUIVO_SEBRAE = f"{VOL}/cadeias/classificacao_cnae_cadeias_v2.xlsx"

print("Volume:", VOL, "| ingestão:", INGESTAO_TS)

## 2. Download dos arquivos da SEFAZ



In [0]:
os.makedirs(f"{VOL}/sefaz", exist_ok=True)
log = []

for tabela, pasta, arquivo, url in FONTES_CSV:
    destino = f"{VOL}/{pasta}/{arquivo}"
    try:
        r = requests.get(url, timeout=120, verify=False,
                         headers={"User-Agent": "Mozilla/5.0"})
        r.raise_for_status()
        with open(destino, "wb") as f:
            f.write(r.content)
        log.append((tabela, arquivo, "OK", len(r.content), ""))
    except Exception as e:
        log.append((tabela, arquivo, "FALHA", 0, f"{type(e).__name__}: {str(e)[:150]}"))

display(spark.createDataFrame(log,
    "tabela string, arquivo string, status string, bytes long, erro string"))

## 3. Detecção de encoding e separador

Os arquivos da SEFAZ não têm encoding uniforme: alguns são UTF-8, outros ISO-8859-1.

In [0]:
def detectar(caminho, n=8192):
    """Devolve (encoding, separador) inferidos do início do arquivo."""
    bruto = open(caminho, "rb").read(n)
    texto, encoding = None, None
    for corte in range(4):
        pedaco = bruto[:len(bruto) - corte] if corte else bruto
        try:
            texto = pedaco.decode("utf-8")
            encoding = "UTF-8"
            break
        except UnicodeDecodeError:
            texto = None
    if texto is None:
        texto = bruto.decode("iso-8859-1")
        encoding = "ISO-8859-1"
    primeira = texto.splitlines()[0]
    sep = ";" if primeira.count(";") > primeira.count(",") else ","
    return encoding, sep


for tabela, pasta, arquivo, _ in FONTES_CSV:
    caminho = f"{VOL}/{pasta}/{arquivo}"
    if not os.path.exists(caminho):
        print(f"--- {tabela}: arquivo ausente\n")
        continue
    encoding, sep = detectar(caminho)
    texto = open(caminho, "rb").read(600).decode(encoding, "ignore")
    print(f"--- {tabela} | encoding={encoding} | sep='{sep}'")
    print("\n".join(texto.splitlines()[:2])[:400], "\n")

print(spark.table("mvp_pipeline_vf.bronze.desoneracoes").columns[:3])

## 4. Ingestão dos CSV da SEFAZ

`limpar_nomes` normaliza os cabeçalhos de todos os arquivos:

- remove acento e espaço (`CNAE Divisão` → `cnae_divisao`), recusados pelo Delta;
- remove o BOM do arquivo de desonerações, gravado em UTF-8 com BOM.

`inferSchema=False`: todas as colunas entram como texto.

In [0]:
def limpar_nomes(df):
    """Normaliza nomes de coluna: sem acento, sem espaço, sem caractere proibido pelo Delta."""
    novos = []
    for c in df.columns:
        n = unicodedata.normalize("NFKD", c).encode("ascii", "ignore").decode()
        n = re.sub(r"[ ,;{}()\n\t=]+", "_", n.strip())
        n = re.sub(r"_+", "_", n).strip("_").lower()
        novos.append(n)
    return df.toDF(*novos)


resultado = []
for tabela, pasta, arquivo, url in FONTES_CSV:
    caminho = f"{VOL}/{pasta}/{arquivo}"
    if not os.path.exists(caminho):
        resultado.append((tabela, 0, 0, "", "", "arquivo ausente"))
        continue
    encoding, sep = detectar(caminho)
    try:
        df = (spark.read
              .option("header", True).option("sep", sep)
              .option("encoding", encoding).option("quote", '"')
              .option("multiLine", False).option("inferSchema", False)
              .csv(caminho))
        df = (limpar_nomes(df)
              .withColumn("_ingestao_ts", lit(INGESTAO_TS))
              .withColumn("_arquivo_origem", lit(arquivo))
              .withColumn("_url_origem", lit(url))
              .withColumn("_fonte", lit("SEFAZ-RS / Receita Dados")))
        (df.write.mode("overwrite").option("overwriteSchema", True)
           .saveAsTable(f"{CATALOGO}.bronze.{tabela}"))
        resultado.append((tabela, df.count(), len(df.columns), encoding, sep, "OK"))
    except Exception as e:
        resultado.append((tabela, 0, 0, encoding, sep, f"{type(e).__name__}: {str(e)[:150]}"))

display(spark.createDataFrame(resultado,
    "tabela string, linhas long, colunas int, encoding string, sep string, status string"))

## 5. IBGE — municípios do RS

API de Localidades, JSON aninhado com microrregião e mesorregião.

In [0]:
municipios = requests.get(
    "https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios",
    timeout=60).json()

with open(f"{VOL}/ibge_municipios_rs.json", "w", encoding="utf-8") as f:
    json.dump(municipios, f, ensure_ascii=False)

(spark.read.option("multiline", True).json(f"{VOL}/ibge_municipios_rs.json")
 .withColumn("_ingestao_ts", lit(INGESTAO_TS))
 .withColumn("_arquivo_origem", lit("ibge_municipios_rs.json"))
 .withColumn("_url_origem", lit("https://servicodados.ibge.gov.br/api/v1/localidades/estados/43/municipios"))
 .withColumn("_fonte", lit("IBGE / API Localidades"))
 .write.mode("overwrite").option("overwriteSchema", True)
 .saveAsTable(f"{CATALOGO}.bronze.ibge_municipios_rs"))

print("Municípios:", len(municipios))

## 6. Sebrae RS — classificação de CNAEs por cadeia produtiva

- `header=1`: a primeira linha é título.
- `.iloc[:, 1:]`: descarta a coluna de numeração.
- `dtype=str`: preserva o zero à esquerda dos códigos CNAE.

In [0]:
import pandas as pd

pdf = pd.read_excel(ARQUIVO_SEBRAE, sheet_name="Classificação CNAEs",
                    header=1, dtype=str).iloc[:, 1:]
pdf.columns = ["cnae_origem", "denominacao_cnae", "cadeia"]
pdf = pdf.dropna(subset=["cnae_origem"])

(spark.createDataFrame(pdf)
 .withColumn("_ingestao_ts", lit(INGESTAO_TS))
 .withColumn("_arquivo_origem", lit("classificacao_cnae_cadeias_v2.xlsx"))
 .withColumn("_url_origem", lit("arquivo institucional, sem URL pública"))
 .withColumn("_fonte", lit("Sebrae RS - Classificação de CNAEs por cadeia produtiva 2026"))
 .write.mode("overwrite").option("overwriteSchema", True)
 .saveAsTable(f"{CATALOGO}.bronze.cnae_cadeia_produtiva"))

print("Linhas:", len(pdf))

## 7. DEE-RS — PIB e PIB per capita municipal

Série histórica 2002–2023 dos 497 municípios, publicada pelo Departamento de Economia e Estatística
do Rio Grande do Sul, conveniado do IBGE na apuração do PIB municipal.



In [0]:
bruto = (spark.read
    .option("sep", ";")
    .option("header", "false")
    .option("multiLine", "true")
    .option("quote", '"')
    .option("escape", '"')
    .option("encoding", "UTF-8")
    .csv(ARQUIVO_DEE))

dee = (bruto
    .filter(F.col("_c0").rlike(r"^\s*\d{4}\s*$"))
    .selectExpr(
        "trim(_c0)  AS ano",
        "trim(_c1)  AS cod_municipio_ibge",
        "trim(_c2)  AS nome_municipio",
        "trim(_c3)  AS vab_agropecuaria",
        "trim(_c4)  AS vab_industria",
        "trim(_c5)  AS vab_servicos",
        "trim(_c6)  AS vab_administracao_publica",
        "trim(_c7)  AS vab_total",
        "trim(_c8)  AS impostos_liquidos",
        "trim(_c9)  AS pib",
        "trim(_c10) AS pib_per_capita")
    .withColumn("_ingestao_ts", lit(INGESTAO_TS))
    .withColumn("_arquivo_origem", lit(os.path.basename(ARQUIVO_DEE)))
    .withColumn("_url_origem", lit("https://dee.rs.gov.br/pib-municipal"))
    .withColumn("_fonte", lit("DEE-RS - PIB dos municipios do RS, serie historica 2002-2023")))

(dee.write.mode("overwrite").option("overwriteSchema", "true")
   .saveAsTable(f"{CATALOGO}.bronze.dee_pib_municipal_rs"))

print("Linhas:", dee.count())

In [0]:
%sql
USE CATALOG mvp_pipeline_vf;

SELECT
  ano,
  count(*)                                        AS municipios,
  sum(CASE WHEN pib_per_capita <> '' THEN 1 END)  AS com_per_capita
FROM bronze.dee_pib_municipal_rs
GROUP BY ano ORDER BY ano;

## 8. Verificação final da camada

Nove tabelas. As contagens são a referência para a reconciliação do notebook 04.

In [0]:
tabelas = [r.tableName for r in spark.sql(f"SHOW TABLES IN {CATALOGO}.bronze").collect()]
contagem = [(t, spark.table(f"{CATALOGO}.bronze.{t}").count()) for t in sorted(tabelas)]
display(spark.createDataFrame(contagem, "tabela string, linhas long"))